In [3]:
import joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

In [ ]:
# Loading the encoded train and test data
train_data = joblib.load('encoded_train_data.joblib', mmap_mode='r')
test_data = joblib.load('encoded_test_data.joblib', mmap_mode='r')
X = train_data.drop(columns=['IncidentGrade'])
y = train_data['IncidentGrade']

# Spliting the data (80:20)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Comparing Machine Learning Models

In [4]:

X_train_subsample = X_train.sample(frac=0.1, random_state=42)
y_train_subsample = y_train.loc[X_train_subsample.index]

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_jobs=-1, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(n_jobs=-1, random_state=42),
    'LightGBM': LGBMClassifier(n_jobs=-1, random_state=42),
}

for model_name, model in models.items():
    print(f'Model: {model_name}')
    
    model.fit(X_train_subsample, y_train_subsample)
    
    y_pred = model.predict(X_val)
    
    # Evaluateing the models
    accuracy = accuracy_score(y_val, y_pred)
    report = classification_report(y_val, y_pred)
    cm = confusion_matrix(y_val, y_pred)
    
    # Displaying the results of the modles
    print(f'Accuracy: {accuracy}')
    print('Classification Report:')
    print(report)
    print('Confusion Matrix:')
    print(cm)
    print('-' * 50)

Model: Logistic Regression
Accuracy: 0.6336040068117593
Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.87      0.71    765560
           1       0.64      0.16      0.26    390976
           2       0.70      0.64      0.67    628025

    accuracy                           0.63   1784561
   macro avg       0.65      0.56      0.55   1784561
weighted avg       0.64      0.63      0.60   1784561

Confusion Matrix:
[[668041  21742  75777]
 [234346  63279  93351]
 [214792  13848 399385]]
--------------------------------------------------
Model: Random Forest
Accuracy: 0.701326544735652
Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.84      0.75    765560
           1       0.67      0.44      0.53    390976
           2       0.75      0.70      0.72    628025

    accuracy                           0.70   1784561
   macro avg       0.70      0.66      0.67   1784561


In [2]:
# Createing a report data
report = {
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'LightGBM', 'Gradient Boosting'],
    'Accuracy': [0.6318, 0.7005, 0.7011, 0.6777, 0.6756, 0.6414],
    'Macro-F1 Score': [0.54, 0.67, 0.67, 0.62, 0.61, 0.55],
    'Precision': [0.64, 0.70, 0.70, 0.71, 0.72, 0.69],
    'Recall': [0.55, 0.66, 0.66, 0.61, 0.61, 0.56]
}

df = pd.DataFrame(report)

print("Comparison Table:")
print(df.to_string(index=False))

best_models_with_max_f1 = df[df['Macro-F1 Score'] == df['Macro-F1 Score'].max()]

if len(best_models_with_max_f1) > 1:
    best_model = best_models_with_max_f1.loc[best_models_with_max_f1['Accuracy'].idxmax()]
else:
    best_model = df.loc[df['Macro-F1 Score'].idxmax()]

print("\nBest Model Based on Macro-F1 Score (and Accuracy in case of a tie):")
print(best_model)

Comparison Table:
              Model  Accuracy  Macro-F1 Score  Precision  Recall
Logistic Regression    0.6318            0.54       0.64    0.55
      Decision Tree    0.7005            0.67       0.70    0.66
      Random Forest    0.7011            0.67       0.70    0.66
            XGBoost    0.6777            0.62       0.71    0.61
           LightGBM    0.6756            0.61       0.72    0.61
  Gradient Boosting    0.6414            0.55       0.69    0.56

Best Model Based on Macro-F1 Score (and Accuracy in case of a tie):
Model             Random Forest
Accuracy                 0.7011
Macro-F1 Score             0.67
Precision                   0.7
Recall                     0.66
Name: 2, dtype: object
